# Lecture 4 — An HR Query Agent

**Goal**: Build an agent that answers questions about a pool of job candidates by **choosing actions**, not just by answering.

In Lectures 2 and 3, every step was decided by *us*: score this resume, then submit the score. Even a multi-step pipeline is a **workflow**: the steps and their order are fixed in our code.

An **agent** is different. We give an LLM a goal and a set of **tools**, and the LLM decides:
- **which** tool to use next,
- **what arguments** to give it, and
- **when it's done**.

Our agent will help an HR department ask questions like *"Which candidates have a master's degree?"* about the candidates who applied for the Senior Full-Stack Software Engineer job. The answer is a table (a Pandas DataFrame) that a human uses to make a decision.

![Overview of the HR query agent](agent_overview.svg)

We'll build it in two stages:
1. An agent with one read-only tool, `respond`, which looks things up in the database.
2. An agent that can also `augment` the database: when the information it needs isn't there, it can ask a fast, cheap LLM to read every resume and add a new column.

## Setup

In [ ]:
import os

OPENROUTER_API_KEY = ""  # <-- Paste your OpenRouter API key here
OPENROUTER_API_KEY = OPENROUTER_API_KEY or os.environ.get("OPENROUTER_API_KEY", "")

# The agent plans, writes queries, and writes prompts, so it should be a strong model.
AGENT_MODEL = "anthropic/claude-sonnet-5.5"

# The augment tool calls a model once per resume, so it should be fast and cheap.
FAST_MODEL = "anthropic/claude-haiku-5.5"

# YOU CAN USE ANY MODEL WITH STRUCTURED OUTPUTS:
# https://openrouter.ai/models?fmt=cards&supported_parameters=structured_outputs

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError("Please set OPENROUTER_API_KEY above")

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

import httpx
import pandas as pd

from resume_utils import load_resumes, load_job_requirements

resumes = load_resumes("../data/resumes_final_with_gold_silver.csv")
job_req = load_job_requirements("../data/job_req_senior.md")

print(f"Loaded {len(resumes)} resumes")

## The Database

Our "database" is a Pandas DataFrame with one row per candidate. The index is `resume_id`, and it starts with the two columns we produced in Lecture 3:
- `score`: how well the resume fits the job, from 0 to 100
- `score_reasoning`: a short explanation of the score

These were computed ahead of time and saved in `../data/lecture_3_scores.csv`, so loading them is instant.

In stage 2, the agent will add columns to this DataFrame. **If you want to start over with a clean database, re-run this cell.**

In [ ]:
def load_database():
    """Load the Lecture 3 scores into a fresh DataFrame, indexed by resume_id."""
    return pd.read_csv(
        "../data/lecture_3_scores.csv", index_col="resume_id", dtype={"resume_id": str}
    )


db = load_database()
db

## Calling the LLM

Every LLM call in this notebook goes through one small function. It sends a list of **messages** to OpenRouter and forces the reply to match a **JSON schema** (structured output), so we always get back a Python dict that we can use directly.

Each message has a `role`:
- `system`: instructions about how to behave (the "job description" for the LLM)
- `user`: the input it should respond to
- `assistant`: something the LLM said earlier in the conversation

The agent will need a whole conversation (many messages), which is why this takes `messages` instead of a single prompt.

In [ ]:
total_cost = 0.0  # running total, in dollars


def call_llm(messages, schema, model):
    """Send messages to OpenRouter; return the JSON object the model produced.

    The reply is forced to match `schema`, a JSON schema, by strict structured output.
    """
    global total_cost

    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.2,
        "max_tokens": 2000,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "output", "strict": True, "schema": schema},
        },
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}

    for attempt in range(3):
        resp = httpx.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120,
        )
        if resp.status_code in (429, 502, 503) and attempt < 2:
            time.sleep(2 * (attempt + 1))  # rate-limited or overloaded: wait, retry
            continue
        resp.raise_for_status()
        data = resp.json()
        if "error" in data:
            raise RuntimeError(f"OpenRouter error: {data['error']}")

        total_cost += data.get("usage", {}).get("cost") or 0
        return json.loads(data["choices"][0]["message"]["content"])

A quick test. The schema says "an object with a `language` string and a `year` number", and that's exactly what comes back:

In [ ]:
call_llm(
    messages=[
        {"role": "user", "content": "Name a programming language and the year it was created."}
    ],
    schema={
        "type": "object",
        "properties": {"language": {"type": "string"}, "year": {"type": "number"}},
        "required": ["language", "year"],
        "additionalProperties": False,
    },
    model=FAST_MODEL,
)

# Stage 1: An Agent with One Tool

## Tool 1: `respond`

A **tool** is just a Python function that the agent is allowed to call. Our first tool, `respond`, answers the HR question by showing part of the database:
- `where` is a [`DataFrame.query`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) expression that picks the **rows**, like `"score >= 90"`
- `select` is a list of the **columns** to show, like `["score", "score_reasoning"]`

`respond` is **read-only**: it never changes the database. It's also the agent's way of saying "I'm done": the DataFrame it returns is the final answer.

Each tool also has a **JSON schema** for its arguments. This is what the agent must fill in to call the tool: here, an object with `select` (a list of at least one string) and `where` (a string).

In [ ]:
def respond(select, where):
    """Read-only: pick rows with a pandas query and columns with a list."""
    # resume_id is the index, so it's always shown; it can't be selected like a column
    select = [column for column in select if column != "resume_id"]

    if where.strip() == "":
        rows = db  # an empty query means "all rows"
    else:
        rows = db.query(where, engine="python")

    return rows[select]


RESPOND_SCHEMA = {
    "type": "object",
    "properties": {
        "select": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "where": {"type": "string"},
    },
    "required": ["select", "where"],
    "additionalProperties": False,
}

We can call it ourselves to see what it does. (`engine="python"` is what lets queries use string methods like `.str.contains`.)

In [ ]:
respond(select=["score", "score_reasoning"], where="score >= 90")

In [ ]:
respond(select=["score"], where='score_reasoning.str.contains("AWS")')

## The Tool Registry

The agent needs to know which tools exist, what they're for, and how to call them. We keep all of that in one dictionary, the `TOOL_REGISTRY`:
- `description`: plain-English instructions for the agent, telling it when and how to use the tool
- `schema`: the JSON schema for the tool's arguments
- `function`: the Python function we run when the agent picks it

To give the agent a new ability, we only have to add an entry here.

In [ ]:
TOOL_REGISTRY = {
    "respond": {
        "description": (
            "Answer the HR question by showing part of the database. `where` is a "
            "pandas DataFrame.query expression that picks the rows to show (use an "
            "empty string for all rows), and `select` is the list of columns to show. "
            "This ends the conversation and its output is your final answer, so only "
            "call it when the database already has what's needed to answer."
        ),
        "schema": RESPOND_SCHEMA,
        "function": respond,
    },
}

## How the Agent Chooses a Tool

On each turn, the agent answers with a structured output that has two parts:
- `tool`: the name of one tool in the registry
- `arguments`: an object matching **one of** the tools' argument schemas (`anyOf`)

We build this schema from the registry, so it grows automatically as we add tools:

In [ ]:
def make_agent_schema(registry):
    """The JSON schema for one agent decision: a tool name and its arguments."""
    return {
        "type": "object",
        "properties": {
            "tool": {"type": "string", "enum": list(registry)},
            "arguments": {"anyOf": [tool["schema"] for tool in registry.values()]},
        },
        "required": ["tool", "arguments"],
        "additionalProperties": False,
    }


print(json.dumps(make_agent_schema(TOOL_REGISTRY), indent=2))

## The Conversation

The agent's whole "memory" is the list of messages that we send it on every turn:

![The agent's message stream and loop](agent_loop.svg)

There's one wrinkle. When a tool runs, we need to tell the agent what happened. We're building tool calls ourselves out of structured output (instead of using the providers' built-in tool-calling features), so after the `system` message the conversation can only contain `user` and `assistant` messages, and they should alternate. So the tool's result is sent as a **`user` message that starts with `TOOL RESULT:`**, and the system prompt explains that these messages come from the tools, not from the HR person.

The system prompt is rebuilt on every turn from the current database, so when a new column appears, the agent sees it.

In [ ]:
def make_system_prompt(registry):
    """Instructions for the agent: the job, the database, the tools, and the rules."""
    columns = "\n".join(f"- {name} ({dtype})" for name, dtype in db.dtypes.items())

    tools = "\n\n".join(
        f"### {name}\n{tool['description']}\nArguments schema: {json.dumps(tool['schema'])}"
        for name, tool in registry.items()
    )

    return f"""You help an HR department answer questions about {len(db)} candidates \
for a job. You can't read the candidates' resumes yourself; you work by calling \
tools on a pandas DataFrame, the database. A human will use your answer to make \
a decision, so be accurate and show the columns that support your answer.

## The job

{job_req}

## The database

The index is `resume_id` (a string like "10089434" or "g01"). The columns are:
{columns}

`score` (0-100) and `score_reasoning` come from an earlier resume-scoring step that \
rated how well each candidate fits the job above.

## Tools

On every turn, choose exactly one tool and give its arguments.

{tools}

## How the conversation works

The first user message is the HR question. After each of your tool calls, you will \
get a user message that starts with "TOOL RESULT:". It was not written by the HR \
person: it is the output of the tool you just called, or an error message if the \
call failed. Use it to decide what to do next.

## Query tips

`where` is run as DataFrame.query(where, engine="python"). Use column names \
directly; `resume_id` refers to the index. String methods work, for example \
`score_reasoning.str.contains("AWS", case=False)`. Missing values are NaN, so add \
`col.notna()` if you need to skip them."""


print(make_system_prompt(TOOL_REGISTRY))

## The Agent Loop

This is the heart of the agent. `ask` takes an HR question and loops:
1. Send the system prompt and the conversation so far to `AGENT_MODEL`, which picks a tool and its arguments.
2. Print what it chose, so we can watch it work.
3. If it chose `respond`, return that DataFrame: we're done.
4. Otherwise, run the tool, add its result to the conversation, and go around again.

If a tool fails (for example, a query with a typo), we don't crash: the error is sent back to the agent as a `TOOL RESULT`, and it can try again.

In [ ]:
def ask(prompt, max_iterations=8):
    """Run the agent on an HR question; return a DataFrame as the answer."""
    messages = [{"role": "user", "content": prompt}]

    for iteration in range(1, max_iterations + 1):
        # 1. The agent decides what to do next
        system = {"role": "system", "content": make_system_prompt(TOOL_REGISTRY)}
        decision = call_llm(
            [system] + messages, make_agent_schema(TOOL_REGISTRY), AGENT_MODEL
        )
        tool, arguments = decision["tool"], decision["arguments"]
        messages.append({"role": "assistant", "content": json.dumps(decision)})

        # 2. Show what it decided
        print(f"Iteration {iteration}: {tool}")
        for name, value in arguments.items():
            print(f"    {name} = {value!r}")

        # 3. Run the tool (errors are reported back to the agent)
        try:
            result = TOOL_REGISTRY[tool]["function"](**arguments)
        except Exception as err:
            print(f"    ERROR: {type(err).__name__}: {err}")
            messages.append(
                {"role": "user", "content": f"TOOL RESULT: {tool} failed: {err!r}"}
            )
            continue

        # 4. respond is the last tool: its output is the answer
        if tool == "respond":
            print(f"\n(total cost so far: ${total_cost:.4f})")
            return result

        messages.append({"role": "user", "content": f"TOOL RESULT: {json.dumps(result)}"})

    raise RuntimeError(f"The agent didn't respond within {max_iterations} iterations")

## Try It: Stage 1

With only `respond` available, the agent's job is to translate an HR question into a query. Watch what it chooses for `where` and `select`.

In [ ]:
ask("Which candidates scored 90 or above?")

In [ ]:
ask("Which candidates have experience with AWS? Show me why.")

Now a question that the database *can't* answer:

In [ ]:
ask("Which candidates have a master's degree?")

The agent did its best, but it can only search `score_reasoning`, which was written to explain the *score*, not to list degrees. The information is in the resumes, but not in the database.

A human analyst in this situation would go read the resumes. In stage 2, we give the agent a tool to do exactly that.

# Stage 2: Letting the Agent Change the Database

## Tool 2: `augment`

`augment` adds a new column to the database (or replaces one). The agent provides:
- `column`: the new column's name, like `"highest_degree"`
- `prompt`: a question to ask about **each** resume, written by the agent itself
- `type`: what kind of answer to expect: `"number"`, `"string"`, or `"date"`

Then `augment` asks `FAST_MODEL` that question about every resume, all 134 of them, in parallel. The answers become the new column, and they're also sent back to the agent as the tool's result, so on its next turn it can decide whether it has enough to `respond` or needs to `augment` again.

![How augment works](augment_flow.svg)

Unlike `respond`, `augment` is **read-write**: it changes the database in place. This is what makes the system a real agent. We never told it which columns to create or what prompts to use; it figures that out from the HR question.

Each `type` becomes a JSON schema for the fast model's answer. Every type also allows `null`, so the model can say "the resume doesn't say" instead of making something up.

In [ ]:
TYPE_SCHEMAS = {
    "number": {"type": ["number", "null"]},
    "string": {"type": ["string", "null"]},
    "date": {
        "type": ["string", "null"],
        "format": "date",
        "description": "an ISO 8601 date: YYYY-MM-DD",
    },
}


def cast_column(values, type):
    """Convert a Series of JSON values into the right Pandas type."""
    if type == "number":
        return pd.to_numeric(values, errors="coerce")
    elif type == "date":
        return pd.to_datetime(values, errors="coerce")
    else:
        return values.astype("string")

`augment_one` asks the question about a single resume. This is the same kind of call as the resume scoring in Lectures 2 and 3, except that the prompt was written by the agent.

In [ ]:
def augment_one(resume_id, prompt, type):
    """Ask FAST_MODEL the agent's question about one resume; return its answer."""
    messages = [
        {
            "role": "system",
            "content": "Answer the question about this resume. "
            "If the resume doesn't contain the answer, answer null.",
        },
        {
            "role": "user",
            "content": f"{prompt}\n\nResume:\n{resumes[resume_id]['Resume_str']}",
        },
    ]
    schema = {
        "type": "object",
        "properties": {"value": TYPE_SCHEMAS[type]},
        "required": ["value"],
        "additionalProperties": False,
    }
    return call_llm(messages, schema, FAST_MODEL)["value"]

`augment` runs `augment_one` on every resume. Calling the LLM 134 times one after another would take minutes, so we use a `ThreadPoolExecutor` to run 16 at a time. `pool.map` works like a `for` loop that runs in parallel.

In [ ]:
def augment(column, prompt, type):
    """Read-write: add (or replace) a database column by asking about every resume."""
    if column == "resume_id":
        raise ValueError("resume_id is the index, and it can't be replaced")

    resume_ids = list(db.index)
    with ThreadPoolExecutor(max_workers=16) as pool:
        values = list(pool.map(lambda rid: augment_one(rid, prompt, type), resume_ids))

    results = dict(zip(resume_ids, values))  # {resume_id: value}
    db[column] = cast_column(pd.Series(results), type)  # changes db in place
    return results


AUGMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "column": {"type": "string"},
        "prompt": {"type": "string"},
        "type": {"type": "string", "enum": list(TYPE_SCHEMAS)},
    },
    "required": ["column", "prompt", "type"],
    "additionalProperties": False,
}

Let's call it ourselves first, the same way we tested `respond`:

In [ ]:
results = augment(
    column="years_experience",
    prompt=(
        "How many years of professional work experience does this person have in "
        "total? Estimate it from the dates in their work history."
    ),
    type="number",
)

db.sort_values("years_experience", ascending=False).head(10)

## Add `augment` to the Registry

The description matters a lot: it's the only thing that tells the agent what the fast model can see. That model gets one resume at a time and nothing else (not the HR question, not the database), so the agent's prompt has to stand on its own.

In [ ]:
TOOL_REGISTRY["augment"] = {
    "description": (
        "Add a new column to the database, or replace an existing one, by asking a "
        "fast LLM the same question about every candidate's full resume text. "
        "`column` is the column's name. `prompt` is the question; the fast LLM sees "
        "only this prompt and one resume, nothing else (not the HR question, not the "
        "database), so make it self-contained and specific. It answers null when the "
        "resume doesn't say. `type` is the kind of answer: number, string, or date "
        "(YYYY-MM-DD). For strings, ask for a short answer from a fixed set of "
        "choices, so the column is easy to query. The tool result is the new "
        "column's value for every resume_id. This is slow and costs money, so reuse "
        "existing columns when they already have what you need."
    ),
    "schema": AUGMENT_SCHEMA,
    "function": augment,
}

print(json.dumps(make_agent_schema(TOOL_REGISTRY), indent=2))

## Try It: Stage 2

Let's ask the question that stumped the stage 1 agent. Watch the iterations: now the agent can decide that it needs more information, go get it, and only then `respond`.

In [ ]:
ask("Which candidates have a master's degree?")

Questions can combine what's already in the database with new information. Notice that the agent can reuse the `years_experience` column we added earlier, or decide that it needs something more specific.

In [ ]:
ask("Of the 10 highest-scoring candidates, which have at least 5 years of C# experience?")

In [ ]:
ask(
    "Among candidates scoring above 70, who started their current or most recent "
    "job before 2015?"
)

Everything the agent added is still in the database, so later questions can use it:

In [ ]:
db

## Summary

What made this an **agent** rather than a workflow?
- **It chose its actions.** The same `ask` function sometimes responds right away and sometimes augments one or more times first, depending on the question.
- **It wrote its own instructions.** The prompts sent to `FAST_MODEL` and the queries sent to Pandas were written by the agent, not by us.
- **It decided when it was done.** The loop ends when the agent calls `respond`, not after a fixed number of steps.
- **It changed its environment.** `augment` writes to the database, and later turns (and later questions) see the result.

Things to try:
- Ask your own HR questions. Which ones does the agent handle well, and which ones confuse it?
- Read the prompts the agent writes for `augment`. Are they specific enough? Compare the new column with a few resumes.
- Add a tool of your own: for example, a `sort_by` argument for `respond`, or a tool that lets the agent look at a few rows before it responds.